## Reading 34-1 - Coding Standards: MISRA C and SEI CERT C

By the end of this reading, you will learn:
<ol>
    <li>What a coding standard is, and why safety-critical teams restrict which parts of C they use</li>
    <li>How MISRA C classifies its rules, and how teams document a justified exception</li>
    <li>How the free SEI CERT C standard maps to the bugs we found in Lecture 32</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading34
    cd reading34

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec34/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    bounded_find.c  Makefile  node_pool.c  recursive_delete.c

The code examples are used in Reading 34-2.

### Using Less of the Language, On Purpose

In Lecture 32, every bug we found was legal C. The compiler accepted it, and most of the time it even produced the right answer. A <b>coding standard</b> takes the opposite approach to "anything legal is allowed": it defines a <i>subset</i> of the language that a team agrees to use, chosen so that the dangerous constructs are forbidden and the remaining code can be checked by tools.

### MISRA C

MISRA C was first published in 1998 by the Motor Industry Software Reliability Association for automotive software. It is now used across automotive, aerospace, rail, and medical devices. The current edition, <b>MISRA C:2025</b>, was released in March 2025 and contains 225 guidelines.

Every guideline is assigned a category:
<ul>
    <li><b>Mandatory:</b> must always be followed, no exceptions</li>
    <li><b>Required:</b> must be followed unless a formal, documented <i>deviation</i> is approved</li>
    <li><b>Advisory:</b> recommended good practice</li>
</ul>

MISRA also labels each rule as <b>decidable</b> (a tool can always determine whether code complies) or <b>undecidable</b> (a tool can only check some cases, so humans must review the rest). That label tells you exactly how much you can trust a "zero violations" report.

The idea of a <b>deviation</b> is as important as the rules. Real projects sometimes need to break a rule. In a MISRA project, you may do so only by writing down which rule you are breaking, why it is necessary, and why it is safe in this specific case, and having that reviewed. An undocumented violation is a defect. A documented, reviewed deviation is engineering.

To give you a sense of the themes, MISRA C rules forbid or restrict:
<ul>
    <li>Allocating memory from the heap with <code>malloc</code> and <code>free</code></li>
    <li>Recursion</li>
    <li>Ignoring the value returned by a non-<code>void</code> function</li>
    <li>Reading a variable before it has been given a value</li>
    <li>Implicit conversions that can lose information, such as storing an <code>int</code> result in a narrower type</li>
</ul>

The MISRA documents are sold, not free, so in this course you will work from a paraphrased subset in Homework 9.

### SEI CERT C

The <a href = "https://cmu-sei.github.io/secure-coding-standards/sei-cert-c-coding-standard/rules">SEI CERT C Coding Standard</a>, from Carnegie Mellon's Software Engineering Institute, is free to read online. Where MISRA asks "will this code behave predictably?", CERT focuses on "can this code be exploited or fail unexpectedly?" Each rule has an identifier such as <code>INT32-C</code>, a rationale, and non-compliant and compliant code examples.

Here is the useful part: <b>every bug from Lecture 32 is already a CERT rule.</b>

<table>
    <tr><th>Bug we saw</th><th>Where</th><th>CERT rule</th></tr>
    <tr><td>Signed overflow folded away by the compiler</td><td><code>signed_overflow.c</code></td><td><code>INT32-C</code>: signed integer operations must not overflow</td></tr>
    <tr><td>17-bit result truncated into a 16-bit variable</td><td><code>promotion.c</code></td><td><code>INT31-C</code>: integer conversions must not lose data</td></tr>
    <tr><td>Writing past the end of the buffer</td><td><code>telemetry.c</code>, <code>full.txt</code></td><td><code>ARR30-C</code>: no out-of-bounds array access</td></tr>
    <tr><td>Averaging values that were never read</td><td><code>telemetry.c</code>, <code>short.txt</code></td><td><code>EXP33-C</code>: do not read uninitialized memory</td></tr>
    <tr><td>Leak on the early-return path</td><td><code>telemetry.c</code>, <code>corrupt.txt</code></td><td><code>MEM31-C</code>: free dynamic memory when no longer needed</td></tr>
    <tr><td>332.5 &deg;C packed into an <code>int16_t</code></td><td><code>telemetry.c</code>, <code>hot.txt</code></td><td><code>FLP34-C</code>: floating-point conversions must be in range</td></tr>
    <tr><td>A clock that drifts by 745 s per day</td><td><code>accumulate.c</code></td><td><code>FLP30-C</code>: do not use floating-point values as loop counters</td></tr>
</table>

The standards were not written by people imagining what might go wrong. They are a record of what already has.

### Where We Go Next

MISRA C has 225 guidelines. In 2006, Gerard Holzmann at NASA's Jet Propulsion Laboratory asked a different question: what is the <i>smallest</i> set of rules that makes code analyzable by tools? His answer was ten rules, known as <b>The Power of Ten</b>, and they became the core of JPL's coding standard for the Mars Science Laboratory. They are the subject of the next reading and Lecture 35.

### <font color = "red">Class Introduction Question #1 - What is the difference between a MISRA <i>required</i> rule and an <i>advisory</i> rule, and what must a team do to break a required rule?</font>

### <font color = "red">Class Introduction Question #2 - Pick one bug from Lecture 32. Name the CERT rule it violates, and explain in one sentence what the compliant version of the code would do differently.</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec34/Reading%2034-2.ipynb">Reading 34-2 - The Power of Ten</a>